# Notebook 01 -- Euroroad Laplacian Renormalization

This notebook reproduces the Laplacian renormalization analysis of the Euroroad
network from the paper "Harmonic morphisms and dynamical invariants in network
renormalization."

**Figures produced:**
- **Figure 2a:** Harmonic degree, conformal degree, and harmonic deviation vs
  compression under Laplacian renormalization.
- **Figure 3:** Spatial distribution of clusters and the coarse-grained graph
  at diffusion time t=4.

In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from harmonic_morphisms import H_CF_curves, renorm_graph_harmonic, g_len
from harmonic_morphisms.core import renorm_graph_plot
from harmonic_morphisms.simplicial import import_network_data

DATA_DIR = Path("../data")

SMALL_SIZE = 10; MEDIUM_SIZE = 12; BIGGER_SIZE = 13
plt.rc("font", size=SMALL_SIZE)
plt.rc("axes", titlesize=SMALL_SIZE)
plt.rc("axes", labelsize=MEDIUM_SIZE)
plt.rc("xtick", labelsize=SMALL_SIZE)
plt.rc("ytick", labelsize=SMALL_SIZE)
plt.rc("legend", fontsize=SMALL_SIZE)
plt.rc("figure", titlesize=BIGGER_SIZE)

In [ ]:
f_road = open(DATA_DIR / "networks" / "out.subelj_euroroad_euroroad")
Ag = import_network_data(f_road)
print(f"Euroroad: {Ag.number_of_nodes()} nodes, {Ag.number_of_edges()} edges")

In [ ]:
L0 = nx.laplacian_matrix(Ag, nodelist=Ag.nodes()).todense()
GRAPHS_l, DEG_H_L, M_DEG_H_l, STD_H_l, AV_H_l, STD_V_H_l, NOT_H_l, \
    DEG_CF_l, M_DEG_CF_l, STD_CF_l, AV_CF_l, STD_V_CF_l, NOT_CF_l, \
    gV_l, t_h_l = H_CF_curves(Ag, L0, 100, pow(10, -4))
laplacian_compression = 1 - np.array(g_len(GRAPHS_l)) / len(Ag.nodes())
print(f"Collapse time: {t_h_l[-1]:.1f}, steps: {len(GRAPHS_l)}")

## Figure 2a: Laplacian harmonic/conformal degree vs compression

In [ ]:
measure_colors = ["salmon", "lightseagreen", "goldenrod"]

f = plt.figure(figsize=(11, 2.5))

ax = plt.subplot(1, 3, 1)
ax.plot(laplacian_compression, M_DEG_H_l, "-o", color=measure_colors[0], markersize=4)
ax.set_xlabel("Compression")
ax.set_ylabel("Mod. harmonic degree")
ax.set_ylim([-0.05, 1.05])
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = plt.subplot(1, 3, 2)
ax.plot(laplacian_compression, M_DEG_CF_l, "-o", color=measure_colors[1], markersize=4)
ax.set_xlabel("Compression")
ax.set_ylabel("Mod. conformal degree")
ax.set_ylim([-0.05, 1.05])
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = plt.subplot(1, 3, 3)
ax.plot(laplacian_compression, STD_H_l, "-o", color=measure_colors[2], markersize=4)
ax.set_xlabel("Compression")
ax.set_ylabel("Harmonic deviation")
ax.set_ylim([-0.05, 0.65])
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.suptitle("Laplacian renormalization on Euroroad network")
plt.tight_layout()
plt.show()

## Figure 3: Spatial distribution of harmonic deviation

In [ ]:
t = 4
G, deg_h, M_deg_h, std_h, av_h, std_v_h, Not_h, \
    deg_cf, M_deg_cf, std_cf, av_cf, std_v_cf, Not_cf, Gv = \
    renorm_graph_harmonic(Ag, t, L0, pow(10, -4))

Values = [deg_h, M_deg_h, std_h, deg_cf, M_deg_cf, std_cf]
Labels = ["H", "Mod. H", "Std. H", "CF", "Mod. CF", "Std. CF"]
df = pd.DataFrame([Values], columns=Labels)

G_plot, colors_d, sing_col = renorm_graph_plot(Ag, t, L0, pow(10, -4))
colors = [colors_d[n] for n in Ag.nodes()]

layout = nx.spring_layout(Ag, iterations=100, seed=42)
lay2 = nx.spring_layout(G_plot, iterations=100, seed=42)

f, ax = plt.subplots(1, 2, figsize=(8.3, 4))
ax = ax.flatten()

nodes = nx.draw_networkx_nodes(Ag, ax=ax[0], pos=layout, node_color=colors, node_size=14)
nodes.set_edgecolor("white")
nx.draw_networkx_edges(Ag, ax=ax[0], pos=layout, width=0.8)
ax[0].collections[0].set_linewidth(0.7)
ax[0].collections[0].set_edgecolor("#FFFFFF")
ax[0].set_title("Clustering", fontsize=10)

nodes_2 = nx.draw_networkx_nodes(G_plot, ax=ax[1], pos=lay2, node_color=sing_col, node_size=60)
nodes_2.set_edgecolor("white")
nx.draw_networkx_edges(G_plot, ax=ax[1], pos=lay2, width=1.2)
ax[1].collections[0].set_linewidth(0.6)
ax[1].collections[0].set_edgecolor("#FFFFFF")
ax[1].set_title(
    f"Laplacian, time {t}, compression {1 - len(G_plot.nodes) / len(Ag.nodes):.2f}",
    fontsize=10,
)

f.suptitle(f"Laplacian Renormalization at time {t}", fontsize=12)
plt.tight_layout()
plt.show()

print(df.to_string(index=False))